## **Part 4: [Exercise]** Rhyme Re-themer Chatbot

Below is a poetry generation example that showcases how you might organize two different tasks under the guise of a single agent. The system calls back to the simple Gradio example, but extends it with some boiler-plate responses and logic behind the scenes.

It's primary feature is as follows:
- On the first response, it will generate a poem based on your response.
- On subsequent responses, it will keep the format and structure of your original rhyme while modifying the topic of the poem.

**Problem:** At present, the system should function just fine for the first part, but the second part is not yet implemented.

**Objective:** Implement the rest of the `rhyme_chat2_stream` method such that the agent is able to function normally.

To make the gradio component easier to reason with, a simplified `queue_fake_streaming_gradio` method is provided that will simulate the gradio chat event loop with the standard Python `input` method

In [ ]:
################################################################################
## SUMMARY OF TASK: chain1 currently gets invoked for the first input.
##  Please invoke chain2 for subsequent invocations.

def rhyme_chat2_stream(message, history, return_buffer=True):
    '''This is a generator function, where each call will yield the next entry'''

    first_poem = None
    for entry in history:
        if entry.get("role") == "assistant":
            content = entry.get("content", "")
            # Logic to extract the poem from previous assistant response
            if "Let me think!" in content:
                # Splits by the "think" preface and the "rewrite" suffix
                parts = content.split("\n\n")
                if len(parts) > 2:
                    first_poem = parts[1]
                    break

    if first_poem is None:
        ## First Case: Generate the initial poem using chain1
        buffer = "Oh! I can make a wonderful poem about that! Let me think!\n\n"
        yield buffer if return_buffer else buffer

        inst_out = ""
        chat_gen = chain1.stream({"input" : message})
        for token in chat_gen:
            inst_out += token
            buffer += token
            yield buffer if return_buffer else token

        passage = "\n\nNow let me rewrite it with a different focus! What should the new focus be?"
        buffer += passage
        yield buffer if return_buffer else passage

    else:
        ## Subsequent Cases: There is a poem to start with. Generate a similar one with a new topic!

        # yield f"Not Implemented!!!"; return ## <- TODO: Comment this out
                
        ########################################################################
        ## TODO: Invoke the second chain to generate the new rhymes.

        buffer = f"Sure! Here you go!\n\n"
        yield buffer
        
        ## iterate over stream generator for second generation
        chat_gen = chain2.stream({"input" : first_poem, "topic" : message})
        for token in chat_gen:
            buffer += token
            yield buffer if return_buffer else token

        ## END TODO
        ########################################################################

        passage = "\n\nThis is fun! Give me another topic!"
        buffer += passage
        yield buffer if return_buffer else passage

################################################################################